# CV03 — Hoàng: Đánh giá XGBoost & Phân tích Feature Importance

**Thành viên phụ trách:** Hoàng  
**Đầu vào:** `data/processed/test.csv` (CV01) + `outputs/models/xgboost_baseline.json` (CV02)  
**Đầu ra:**  
- `outputs/tables/cv03_evaluation_metrics.csv`  
- `outputs/figures/cv03_actual_vs_predicted.png`  
- `outputs/figures/cv03_feature_importance.png`

---

## 1. Cấu hình

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline

from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

NOTEBOOK_DIR = Path().resolve()
PROJECT_ROOT = NOTEBOOK_DIR.parent
TEST_DATA    = PROJECT_ROOT / 'data'    / 'processed' / 'test.csv'
MODEL_PATH   = PROJECT_ROOT / 'outputs' / 'models'    / 'xgboost_baseline.json'
FIGURES_DIR  = PROJECT_ROOT / 'outputs' / 'figures'
TABLES_DIR   = PROJECT_ROOT / 'outputs' / 'tables'

TARGET_COL  = 'AEP_MW'
ZOOM_HOURS  = 168  # 1 tuần để zoom biểu đồ
print('✅ Cấu hình hoàn tất')

## 2. Nạp test data & model

In [ ]:
test_df = pd.read_csv(TEST_DATA)
test_df['Datetime'] = pd.to_datetime(test_df['Datetime'])
test_df.set_index('Datetime', inplace=True)

model = XGBRegressor()
model.load_model(str(MODEL_PATH))

feature_names = model.get_booster().feature_names
X_test = test_df[feature_names]
y_test = test_df[TARGET_COL]

print(f'Test size : {X_test.shape}')
print(f'Features  : {len(feature_names)}')

## 3. Dự báo & Tính chỉ số đánh giá (Bước 6 đề bài)

In [ ]:
y_pred = model.predict(X_test)
y_true = y_test.to_numpy()

mae  = mean_absolute_error(y_true, y_pred)
rmse = np.sqrt(mean_squared_error(y_true, y_pred))
mape = np.mean(np.abs((y_true - y_pred) / y_true)) * 100
r2   = r2_score(y_true, y_pred)

print(f'MAE   = {mae:,.2f} MW')
print(f'RMSE  = {rmse:,.2f} MW')
print(f'MAPE  = {mape:.3f} %')
print(f'R²    = {r2:.6f}')

metrics_df = pd.DataFrame([{
    'Mô hình': 'XGBoost Baseline',
    'MAE (MW)': round(mae, 4),
    'RMSE (MW)': round(rmse, 4),
    'MAPE (%)': round(mape, 4),
    'R2 Score': round(r2, 6),
}])
metrics_df.to_csv(TABLES_DIR / 'cv03_evaluation_metrics.csv', index=False, encoding='utf-8')
print('✅ Đã lưu cv03_evaluation_metrics.csv')
metrics_df

## 4. Biểu đồ Actual vs Predicted

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 9))
plt.subplots_adjust(hspace=0.4)

# Toàn bộ tập test
ax1 = axes[0]
ax1.plot(test_df.index, y_true, label='Thực tế',  color='#1f77b4', lw=0.5, alpha=0.8)
ax1.plot(test_df.index, y_pred, label='Dự báo',   color='#ff7f0e', lw=0.5, alpha=0.8)
ax1.set(title=f'Actual vs Predicted — toàn tập test  (MAE={mae:.0f} MW, RMSE={rmse:.0f} MW, R²={r2:.4f})',
        ylabel='Tiêu thụ điện (MW)')
ax1.legend(); ax1.grid(linestyle=':', alpha=0.5)

# Zoom 168 giờ (1 tuần đầu test)
ax2 = axes[1]
idx_zoom = test_df.index[:ZOOM_HOURS]
ax2.plot(idx_zoom, y_true[:ZOOM_HOURS], label='Thực tế', color='#1f77b4', lw=1.2, marker='.')
ax2.plot(idx_zoom, y_pred[:ZOOM_HOURS], label='Dự báo',  color='#ff7f0e', lw=1.2, marker='.', ls='--')
ax2.set(title=f'Zoom: 168 giờ đầu tập test ({idx_zoom.min().date()} → {idx_zoom.max().date()})',
        ylabel='Tiêu thụ điện (MW)')
ax2.legend(); ax2.grid(linestyle=':', alpha=0.5)

fig.savefig(FIGURES_DIR / 'cv03_actual_vs_predicted.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Đã lưu cv03_actual_vs_predicted.png')

## 5. Phân tích Feature Importance (Bước 7 đề bài)

In [ ]:
importance = model.feature_importances_
fi_df = pd.DataFrame({'feature': feature_names, 'importance': importance})
fi_df = fi_df.sort_values('importance', ascending=False).reset_index(drop=True)
fi_df['rank'] = fi_df.index + 1
fi_df.to_csv(TABLES_DIR / 'cv03_feature_importance.csv', index=False, encoding='utf-8')

fig, ax = plt.subplots(figsize=(10, 6))
colors = ['#d62728' if i < 3 else '#1f77b4' for i in range(len(fi_df))]
ax.barh(fi_df['feature'][::-1], fi_df['importance'][::-1], color=colors[::-1])
ax.set(title='Feature Importance — XGBoost Baseline (gain)', xlabel='Importance Score')
ax.grid(axis='x', linestyle=':', alpha=0.5)
fig.tight_layout()
fig.savefig(FIGURES_DIR / 'cv03_feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Đã lưu cv03_feature_importance.png')

print('\n🏆 Top 5 đặc trưng quan trọng nhất:')
print(fi_df[['rank','feature','importance']].head())

## 6. Nhận xét & Giải thích kết quả

### 6.1 Chất lượng dự báo
- **MAE** cho biết sai số trung bình tuyệt đối tính bằng MW — giá trị càng thấp càng tốt.
- **RMSE** phạt nặng các sai số lớn (outlier) do bình phương hóa.
- **MAPE** biểu diễn sai số dưới dạng phần trăm, dễ hiểu với người dùng cuối.
- **R²** gần 1.0 nghĩa là mô hình giải thích được phần lớn phương sai của tập test.

### 6.2 Feature Importance
- `lag_1` thường dẫn đầu → giá trị tiêu thụ 1 giờ trước có tương quan rất cao với hiện tại.
- `lag_24`, `lag_168` có điểm cao → chu kỳ ngày và tuần rõ ràng trong dữ liệu điện lực.
- `hour` quan trọng vì tải điện thay đổi đáng kể theo khung giờ trong ngày.

## ✅ CV03 Hoàn tất

| Sản phẩm | Đường dẫn |
|:---|:---|
| Bảng chỉ số | `outputs/tables/cv03_evaluation_metrics.csv` |
| Actual vs Predicted | `outputs/figures/cv03_actual_vs_predicted.png` |
| Feature Importance | `outputs/figures/cv03_feature_importance.png` |